# 🏥 Baseline Exploration Notebook - Medical Document Retrieval (R2AI 2026)

Notebook này hỗ trợ các thành viên trong đội:
1. Khám phá và kiểm tra pipeline tiền xử lý & phân đoạn (Chunking)
2. Thử nghiệm trích xuất đặc trưng Dense (BGE-M3) và BM25 Sparse Search
3. Kiểm tra tính năng Reciprocal Rank Fusion (RRF) & Reranking
4. Tính toán metric Macro F2 theo đặc tả của cuộc thi

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

from src.config import load_config
from src.evaluation.metrics import compute_prf
from src.ingestion.chunker import DocumentChunker

config = load_config("../configs/config.yaml")
print("Loaded configuration:", config)

## 1. Thử nghiệm Chunking tài liệu đa ngôn ngữ

In [ ]:
chunker = DocumentChunker(
    max_chunk_size=config.chunking.max_chunk_size,
    chunk_overlap=config.chunking.chunk_overlap,
    min_chunk_size=config.chunking.min_chunk_size,
)

sample_doc = """Bệnh sỏi thận (sỏi niệu) là sự hình thành các tinh thể khoáng chất trong thận.
Khi kích thước sỏi tăng lên, nó có thể di chuyển và gây tắc nghẽn đường dẫn nước tiểu (niệu quản),
dẫn tới cơn đau quặn thận dữ dội, tiểu ra máu hoặc nhiễm trùng đường tiết niệu.
Các phương pháp điều trị bao gồm: uống nhiều nước, dùng thuốc giãn cơ trơn,
tán sỏi ngoài cơ thể (ESWL) hoặc nội soi tán sỏi qua da."""

chunks = chunker.chunk_document(doc_id="sample_vi_1", text=sample_doc, lang="vi")
print(f"Total chunks generated: {len(chunks)}")
for c in chunks:
    print(f"- [{c.chunk_id}] (len={len(c.chunk_text)}): {c.chunk_text[:100]}...")

## 2. Kiểm tra tính điểm Macro F2 (Beta = 2)

In [ ]:
retrieved_docs = ["doc_1", "doc_2", "doc_3"]
ground_truth_docs = ["doc_1", "doc_2", "doc_4"]

p, r, f2 = compute_prf(set(retrieved_docs), set(ground_truth_docs), beta=2.0)
print(f"Precision: {p:.4f}")
print(f"Recall:    {r:.4f}")
print(f"F2 Score:  {f2:.4f}")